# 11 — Hybrid Aggregation Predictor

Combines ESM-2 embeddings (PCA-reduced) + flow model structural features + ESM-2 LLR
into a single GradientBoosting model for aggregation rate prediction.

**Goal:** Bridge the performance gap between:
- Flow ΔRg only: ρ ≈ 0.23
- ESM-2 embedding probe: ρ ≈ 0.66
- **Hybrid target: ρ ≥ 0.45**

**Runtime:** ~15 min on Colab T4 (mostly flow ensemble generation)

In [ ]:
# Cell 0: Setup — 반드시 런타임 재시작 후 이 셀부터 실행!
import os, sys

if 'google.colab' in sys.modules:
    os.chdir('/content')  # 항상 /content 에서 시작

    if not os.path.isdir('/content/brain-idp-flow/src'):
        # 기존 잔재 제거 후 새로 clone
        !rm -rf /content/brain-idp-flow /content/brain_idp_flow
        !git clone https://github.com/layeredlabs06/brain-idp-flow.git /content/brain-idp-flow

    os.chdir('/content/brain-idp-flow')
    sys.path.insert(0, '/content/brain-idp-flow/src')
    !pip install -e '.[esm,ml]' -q
else:
    repo = os.path.expanduser('~/workspace/brain_idp_flow')
    os.chdir(repo)
    src_path = os.path.join(repo, 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

# 검증
from brain_idp_flow.data.dms_loader import ABETA42_WT
print(f'OK — cwd={os.getcwd()}, sequence length={len(ABETA42_WT)}')

import torch, numpy as np
from scipy.stats import spearmanr

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load DMS Data

In [ ]:
from brain_idp_flow.data.dms_loader import load_seuma_dms, ABETA42_WT

dms_data = load_seuma_dms()
print(f'Loaded {len(dms_data)} Aβ42 mutations')
print(f'WT sequence ({len(ABETA42_WT)} aa): {ABETA42_WT}')

# Preview
for d in dms_data[:3]:
    print(f"  {d['mutation_id']}: nscore={d['nucleation_score']:.2f}, agg_rate={d['agg_rate']:.2f}")

## 2. Compute ESM-2 Embeddings

In [ ]:
from brain_idp_flow.features.seq_embed import ESM2Embedder

embedder = ESM2Embedder(model_name='esm2_t12_35M_UR50D', device=DEVICE)

# WT embedding
wt_emb = embedder.embed_single(ABETA42_WT)  # (L, 480)
print(f'WT embedding: {wt_emb.shape}')

# Mutant embeddings — mean-pooled for hybrid model input
mut_embeddings = []  # (N, 480)
for d in dms_data:
    seq = list(ABETA42_WT)
    seq[d['pos'] - 1] = d['mt']
    mut_seq = ''.join(seq)
    emb = embedder.embed_single(mut_seq)  # (L, 480)
    mut_embeddings.append(emb.mean(dim=0).cpu().numpy())  # mean pool -> (480,)

mut_embeddings = np.array(mut_embeddings)
print(f'Mutant embeddings: {mut_embeddings.shape}')

## 3. Compute ESM-2 LLR

In [ ]:
from brain_idp_flow.features.esm2_llr import compute_llr_single

llr_values = []
for d in dms_data:
    llr = compute_llr_single(
        embedder._model, embedder._alphabet,
        ABETA42_WT, d['pos'], d['mt'], device=DEVICE
    )
    llr_values.append(llr)

llr_values = np.array(llr_values)
rho_llr, p_llr = spearmanr(llr_values, [d['nucleation_score'] for d in dms_data])
print(f'ESM-2 LLR vs nucleation: ρ={rho_llr:.3f}, p={p_llr:.4f}')

## 4. Generate Flow Ensembles & Extract Structural Features

This is the slow step (~10 min). We generate WT + each mutant ensemble,
then extract geometry features.

In [ ]:
import yaml
from brain_idp_flow.sample import load_model, sample_ensemble
from brain_idp_flow.model.hybrid_predictor import EnsembleFeatureExtractor

# Load trained flow model
config = yaml.safe_load(open('configs/flow.yaml'))
CKPT_PATH = 'runs/flow_35m/best.pt'  # adjust to your checkpoint path

model = load_model(config, CKPT_PATH, DEVICE)
print('Flow model loaded')

# WT ensemble
wt_ensemble = sample_ensemble(
    model, wt_emb, mut_pos=0, mut_aa=0,
    n_samples=200, n_steps=50, device=DEVICE
)
print(f'WT ensemble: {wt_ensemble.shape}')

In [ ]:
from tqdm.auto import tqdm

AA_TO_IDX = {aa: i + 1 for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}

extractor = EnsembleFeatureExtractor()
structural_features = []

for d in tqdm(dms_data, desc='Generating mutant ensembles'):
    mut_pos = d['pos']
    mut_aa_idx = AA_TO_IDX.get(d['mt'], 0)

    # Apply mutation to sequence and get embedding
    seq = list(ABETA42_WT)
    seq[mut_pos - 1] = d['mt']
    mut_seq = ''.join(seq)
    mut_emb = embedder.embed_single(mut_seq)

    # Generate mutant ensemble
    mut_ensemble = sample_ensemble(
        model, mut_emb, mut_pos=mut_pos, mut_aa=mut_aa_idx,
        n_samples=200, n_steps=50, device=DEVICE, batch_size=64
    )

    # Extract features
    sf = extractor.extract(wt_ensemble, mut_ensemble, mut_pos)
    structural_features.append(sf)

print(f'Extracted features for {len(structural_features)} mutations')
print(f'Feature keys: {list(structural_features[0].keys())}')

## 5. Train Hybrid Model (5-fold CV)

In [ ]:
from brain_idp_flow.model.hybrid_predictor import HybridAggregationPredictor

targets = np.array([d['nucleation_score'] for d in dms_data])

predictor = HybridAggregationPredictor(n_pca=50, n_folds=5)
results = predictor.fit(
    embeddings=mut_embeddings,
    structural_features=structural_features,
    llr_values=llr_values,
    targets=targets,
)

## 6. Ablation: Compare Model Variants

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

def cv_rho(X, y, n_folds=5):
    """Quick 5-fold CV Spearman rho for GBRegressor."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    rhos = []
    for tr, te in kf.split(X):
        sc = StandardScaler()
        gb = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                       min_samples_leaf=5, random_state=42)
        gb.fit(sc.fit_transform(X[tr]), y[tr])
        pred = gb.predict(sc.transform(X[te]))
        r, _ = spearmanr(pred, y[te])
        if not np.isnan(r):
            rhos.append(r)
    return np.mean(rhos) if rhos else 0.0

# Prepare feature variants
pca = PCA(n_components=min(50, len(dms_data)-1), random_state=42)
emb_pca = pca.fit_transform(mut_embeddings)

struct_keys = list(structural_features[0].keys())
struct_matrix = np.array([[sf[k] for k in struct_keys] for sf in structural_features])

y = targets

# Ablation table
variants = {
    'ΔRg only': struct_matrix[:, struct_keys.index('delta_rg')].reshape(-1, 1),
    'LLR only': llr_values.reshape(-1, 1),
    'Structural only': struct_matrix,
    'Embedding PCA only': emb_pca,
    'Embedding + LLR': np.hstack([emb_pca, llr_values.reshape(-1, 1)]),
    'Structural + LLR': np.hstack([struct_matrix, llr_values.reshape(-1, 1)]),
    'Hybrid (all)': np.hstack([emb_pca, struct_matrix, llr_values.reshape(-1, 1)]),
}

print(f"{'Variant':<25} {'CV ρ':>8} {'Features':>10}")
print('-' * 45)
for name, X_var in variants.items():
    rho = cv_rho(X_var, y)
    print(f'{name:<25} {rho:>8.3f} {X_var.shape[1]:>10}')

## 7. IAPP Out-of-Distribution Validation

Train on Aβ42 DMS, evaluate on IAPP mutations from targets.yaml.

In [ ]:
# Load IAPP targets
from brain_idp_flow.targets import load_targets

targets_cfg = load_targets('configs/targets.yaml')
iapp_target = None
for t in targets_cfg:
    # Look for a target with known aggregation rates we can test against
    if t.id in ('asyn', 'tau_k18'):
        iapp_target = t
        break

if iapp_target is not None:
    print(f'OOD target: {iapp_target.name} ({len(iapp_target.mutations)} mutations)')

    # Generate embeddings and ensembles for OOD mutations
    ood_embeddings = []
    ood_llr = []
    ood_struct = []
    ood_targets = []

    wt_seq = iapp_target.sequence
    wt_emb_ood = embedder.embed_single(wt_seq)
    wt_ens_ood = sample_ensemble(
        model, wt_emb_ood, mut_pos=0, mut_aa=0,
        n_samples=200, n_steps=50, device=DEVICE
    )

    for mut in tqdm(iapp_target.mutations, desc=f'OOD: {iapp_target.name}'):
        if mut.agg_rate_relative is None:
            continue

        mut_seq = iapp_target.mutant_sequence(mut)
        mut_emb_ood = embedder.embed_single(mut_seq)
        ood_embeddings.append(mut_emb_ood.mean(dim=0).cpu().numpy())

        # LLR
        from brain_idp_flow.features.esm2_llr import compute_llr_single
        llr = compute_llr_single(
            embedder._model, embedder._alphabet,
            wt_seq, mut.pos, mut.mt, device=DEVICE
        )
        ood_llr.append(llr)

        # Ensemble + features
        mut_aa_idx = AA_TO_IDX.get(mut.mt, 0)
        mut_ens = sample_ensemble(
            model, mut_emb_ood, mut_pos=mut.pos, mut_aa=mut_aa_idx,
            n_samples=200, n_steps=50, device=DEVICE, batch_size=64
        )
        sf = extractor.extract(wt_ens_ood, mut_ens, mut.pos)
        ood_struct.append(sf)
        ood_targets.append(np.log(mut.agg_rate_relative + 1e-8))

    if ood_targets:
        ood_result = predictor.evaluate_ood(
            embeddings=np.array(ood_embeddings),
            structural_features=ood_struct,
            llr_values=np.array(ood_llr),
            targets=np.array(ood_targets),
            dataset_name=iapp_target.name,
        )
else:
    print('No suitable OOD target found in targets.yaml')

## 8. Feature Contribution Visualization

In [ ]:
import matplotlib.pyplot as plt

# Pick a fAD mutation for explanation
fad_idx = next((i for i, d in enumerate(dms_data) if d.get('is_fad')), 0)
d = dms_data[fad_idx]
print(f"Explaining: {d['mutation_id']} (fAD={d.get('is_fad')})")

contributions = predictor.explain(
    embedding=mut_embeddings[fad_idx],
    structural_features=structural_features[fad_idx],
    llr_value=llr_values[fad_idx],
)

# Plot top 15 contributions
top = contributions[:15]
names = [c.name for c in top]
vals = [c.contribution for c in top]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d73027' if v > 0 else '#4575b4' for v in vals]
ax.barh(range(len(top)), vals, color=colors)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Feature Contribution')
ax.set_title(f"Feature Contributions: {d['mutation_id']}")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## 9. Summary Table

In [ ]:
print('\n' + '='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f"{'Method':<30} {'Aβ42 CV ρ':>12} {'OOD ρ':>10}")
print('-'*55)
print(f"{'Flow ΔRg only':<30} {'~0.23':>12} {'~0.11':>10}")
print(f"{'ESM-2 embedding probe':<30} {'~0.66':>12} {'~0.53':>10}")
print(f"{'Cross-scale (GB)':<30} {'~0.35':>12} {'~0.12':>10}")
if results:
    ood_rho = f"{ood_result['rho']:.3f}" if 'ood_result' in dir() and ood_result else 'N/A'
    print(f"{'Hybrid (this notebook)':<30} {results['cv_mean_rho']:>12.3f} {ood_rho:>10}")
print('\nDone!')